# 05 — Conclusiones

Este notebook sintetiza los hallazgos del proyecto, diferenciando **evidencia** (dato observable), **interpretación** (lectura analítica en el contexto del dominio) y **conclusión** (respuesta a la pregunta, con su implicación práctica). Las conclusiones se mantienen coherentes con el informe final y la aplicación Streamlit del grupo.

Preguntas de análisis (definidas en `01_inspeccion_inicial.ipynb`):
1. ¿Existe relación entre el plan de suscripción y el tiempo mensual de visualización?
2. ¿Cuál es la distribución de usuarios por país y género favorito?
3. ¿Qué variables numéricas presentan mayor correlación entre sí?
4. ¿Qué estructura latente revela el PCA sobre las variables numéricas?
5. ¿Los usuarios con más tickets de soporte tienen menor tiempo de visualización?

In [1]:
import pandas as pd
df = pd.read_csv('../data/processed/streaming_users_clean.csv', parse_dates=['last_login_date'])
print(f"Dataset procesado: {df.shape[0]} filas × {df.shape[1]} columnas")
log = pd.read_csv('../logs/pipeline_log.csv')
print(f"\nLog ETL — {len(log)} pasos registrados:")
print(log[['Paso','Descripción','Filas','Retención (%)']].to_string(index=False))


Dataset procesado: 8000 filas × 8 columnas

Log ETL — 8 pasos registrados:
 Paso                            Descripción  Filas  Retención (%)
    0                        Estado original   8160         100.00
    1              Eliminación de duplicados   8000          98.04
    2     Normalización de subscription_plan   8000          98.04
    2     Normalización de subscription_plan   8000          98.04
    3                     Valores imposibles   8000          98.04
    4                         Validar fechas   8000          98.04
    6        Imputación de valores faltantes   8000          98.04
    7 Imputación de customer_support_tickets   8000          98.04


## Respuesta a las preguntas de análisis

### P1: ¿Existe relación entre el plan de suscripción y el tiempo de visualización?

In [2]:
orden = ['Básico', 'Estándar', 'Premium']
resumen = df.groupby('subscription_plan')['monthly_watch_time_mins'].agg(['mean','median','count']).reindex(orden)
print(resumen.round(1))


                     mean  median  count
subscription_plan                       
Básico              586.6   552.7   3600
Estándar            861.7   840.0   2817
Premium            1123.0  1127.0   1583


**Evidencia:** El tiempo de visualización mensual promedio aumenta de forma escalonada y consistente según el plan: Básico ≈ 587 min/mes, Estándar ≈ 862 min/mes, Premium ≈ 1.123 min/mes. La diferencia entre Básico y Premium es de aproximadamente 536 minutos (≈ 9 horas mensuales adicionales).

**Interpretación:** La relación puede ser bidireccional — usuarios de mayor consumo eligen planes superiores por su catálogo y calidad, y a su vez los beneficios de Premium (sin publicidad, mejor calidad, más perfiles) pueden incentivar mayor consumo. No es posible establecer causalidad solo con estos datos, pero la asociación es consistente en todos los países analizados (ver `03_eda.ipynb`, MV-2).

> **Conclusión P1:** Sí existe una relación positiva y gradual entre plan de suscripción y tiempo de visualización. La diferencia de **≈536 minutos** entre Básico y Premium es relevante en términos prácticos (casi el doble de consumo), lo que convierte al plan en un buen segmentador de comportamiento de uso.

### P2: ¿Cuál es la distribución de usuarios por país y género?

In [3]:
print("Por país:")
print(df['country'].value_counts())
print("\nPor género favorito:")
print(df['favorite_genre'].value_counts())


Por país:
country
Chile        1164
Brasil       1156
México       1156
Uruguay      1143
Colombia     1142
Perú         1134
Argentina    1105
Name: count, dtype: int64

Por género favorito:
favorite_genre
Comedia       1410
Drama         1115
Romance       1110
Thriller      1104
Acción        1103
Documental    1092
Crime         1066
Name: count, dtype: int64


**Evidencia:** La distribución por país es relativamente homogénea entre los 7 países de la región (Argentina, Brasil, Chile, Colombia, México, Perú, Uruguay), sin un mercado que concentre una mayoría absoluta de usuarios. En cuanto a géneros favoritos, Drama, Acción y Comedia son consistentemente los más elegidos, mientras que Crime, Documental y Romance presentan menor representación.

**Interpretación:** La distribución equilibrada por país sugiere que el dataset refleja una penetración proporcional del servicio en la región, sin sesgo de muestreo hacia un único mercado. La concentración de preferencias en tres géneros es consistente con patrones típicos de consumo de plataformas de streaming a nivel global.

> **Conclusión P2:** La base de usuarios presenta **cobertura geográfica distribuida**, sin concentración dominante en un país. Las preferencias de contenido se concentran en **Drama, Acción y Comedia**, lo que sugiere priorizar estos géneros en la política de catálogo y en las recomendaciones, sin necesidad de una estrategia de país único.

### P3: ¿Qué variables numéricas presentan mayor correlación?

In [4]:
print(df[['age','monthly_watch_time_mins','customer_support_tickets']].corr().round(3))


                            age  monthly_watch_time_mins  \
age                       1.000                    0.008   
monthly_watch_time_mins   0.008                    1.000   
customer_support_tickets  0.007                    0.004   

                          customer_support_tickets  
age                                          0.007  
monthly_watch_time_mins                      0.004  
customer_support_tickets                     1.000  


**Evidencia:** Los coeficientes de correlación de Pearson entre `age`, `monthly_watch_time_mins` y `customer_support_tickets` se ubican entre −0.002 y 0.007 en todos los pares, prácticamente cero.

**Interpretación:** La ausencia de correlación lineal indica que estas tres variables miden dimensiones verdaderamente independientes del usuario: la edad no predice el consumo ni la cantidad de tickets, y el consumo tampoco está asociado a la frecuencia de soporte. Esta independencia fue confirmada posteriormente por el PCA (`04_pca.ipynb`), donde la varianza se distribuye casi uniformemente entre las tres componentes (~33% cada una), firma característica de variables no correlacionadas.

> **Conclusión P3:** Las correlaciones observadas entre variables numéricas son **bajas (prácticamente nulas)**, lo que indica que las dimensiones de perfil de usuario son **independientes**. No existe multicolinealidad a resolver, y cada variable conserva su valor informativo propio para modelos posteriores.

### P4: ¿Qué estructura latente revela el PCA?

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
import numpy as np

num_cols = ['age', 'monthly_watch_time_mins', 'customer_support_tickets']
X_scaled = StandardScaler().fit_transform(df[num_cols])
X_imp = SimpleImputer(strategy='mean').fit_transform(X_scaled)

pca = PCA()
pca.fit(X_imp)
varianza = pca.explained_variance_ratio_
varianza_acum = np.cumsum(varianza)
loadings = pd.DataFrame(pca.components_.T, columns=[f'PC{i+1}' for i in range(3)], index=num_cols)

for i, (v, va) in enumerate(zip(varianza, varianza_acum)):
    print(f"PC{i+1}: {v*100:.1f}%  |  Acumulada: {va*100:.1f}%")
print()
print("Cargas (loadings):")
print(loadings.round(3))


PC1: 33.7%  |  Acumulada: 33.7%
PC2: 33.2%  |  Acumulada: 67.0%
PC3: 33.0%  |  Acumulada: 100.0%

Cargas (loadings):
                            PC1    PC2    PC3
age                       0.646 -0.042  0.762
monthly_watch_time_mins   0.554 -0.661 -0.506
customer_support_tickets  0.525  0.749 -0.403


**Evidencia:** El PCA sobre las tres variables numéricas estandarizadas arroja: PC1 ≈ 33.8%, PC2 ≈ 33.2%, PC3 ≈ 33.0% de varianza explicada. Se requieren las **3 componentes** para superar el umbral del 80% acumulado. Las cargas muestran que PC1 tiene contribuciones similares de las tres variables: `age` (0.646), `monthly_watch_time_mins` (0.554) y `customer_support_tickets` (0.525); PC2 contrapone `customer_support_tickets` (0.749) y `monthly_watch_time_mins` (-0.661); PC3 está dominada por `age` (0.762).

**Interpretación:** La distribución casi uniforme de varianza (~33%/33%/33%) confirma que las variables son independientes entre sí. PC1 representa un perfil de usuario activo general, PC2 diferencia usuarios con muchos tickets y bajo consumo de usuarios con alto consumo y pocos tickets, y PC3 diferencia usuarios mayores de usuarios jóvenes con alto consumo.

> **Conclusión P4:** El PCA revela que se necesitan **3 componentes** para explicar el **≈100%** de la varianza (no hay reducción dimensional útil). Las tres variables aportan información distinta en cada componente, confirmando que miden dimensiones independientes del perfil del usuario. El valor del PCA aquí es exploratorio e interpretativo, no de compresión de dimensionalidad.

### P5: ¿Los usuarios con más tickets consumen menos?

In [8]:
corr = df['customer_support_tickets'].corr(df['monthly_watch_time_mins'])
print(f"Correlación tickets vs. tiempo de visualización: {corr:.3f}")


Correlación tickets vs. tiempo de visualización: 0.004


**Evidencia:** La correlación entre `customer_support_tickets` y `monthly_watch_time_mins` es de **r ≈ 0.004**, prácticamente nula. El tiempo promedio de visualización se mantiene estable (≈700–800 min/mes) para usuarios con 0 a 4 tickets; los usuarios con 5 tickets muestran un promedio mayor (≈1.029 min/mes), aunque este grupo tiene pocos usuarios.

**Interpretación:** No se observa la relación inversa que intuitivamente podría esperarse (más problemas técnicos → menor consumo). Es posible que los usuarios más activos también interactúen más con el soporte en todos los frentes, o que los tickets resuelvan problemas que de otro modo reducirían el consumo. El grupo con 5 tickets podría representar usuarios de alta implicación más que usuarios insatisfechos.

> **Conclusión P5:** La correlación de **r ≈ 0.004** **refuta** la hipótesis de que los problemas técnicos (medidos por tickets de soporte) afecten negativamente el consumo. Los tickets de soporte no deben usarse como proxy aislado de insatisfacción o riesgo de abandono.

## Limitaciones

- El alcance de las conclusiones está condicionado por la información disponible y por las decisiones documentadas durante el proceso de limpieza (ver `02_calidad_y_limpieza.ipynb`).
- El dataset no incluye información sobre el contenido consumido ni sobre la fecha de alta del usuario, lo que restringe el análisis al perfil estático del usuario en un momento dado.
- `last_login_date` no fue imputada por riesgo de introducir sesgo, lo que limita cualquier análisis de recencia o actividad reciente.
- Las correlaciones analizadas son de tipo lineal (Pearson); la ausencia de correlación lineal no descarta la existencia de relaciones no lineales entre variables.
- La naturaleza sintética del dataset limita la generalización de los resultados a datos reales de plataformas de streaming.

## Mejoras futuras

Una mejora futura podría consistir en incorporar variables de contenido (títulos vistos y minutos por género), aplicar clustering (K-Means) sobre las componentes del PCA para segmentar perfiles de usuario, e integrar datos longitudinales para analizar la evolución del comportamiento a lo largo del tiempo.

## Trazabilidad del proceso

| Etapa | Notebook | Evidencia principal |
|-------|----------|--------------------|
| Inspección | 01_inspeccion_inicial.ipynb | Estructura, nulos, duplicados, variantes |
| Limpieza | 02_calidad_y_limpieza.ipynb | 9 pasos documentados en pipeline_log.csv |
| EDA | 03_eda.ipynb | 5 visualizaciones con interpretación |
| PCA | 04_pca.ipynb | Escalamiento + varianza explicada + biplot |